# The Safety Net — Three Tables, Three Populations

"What share of poor households receive SNAP?" is the natural question to ask of this data, and
it is the one question the ACS cannot answer. Not because the numbers are imprecise — they are
unusually solid — but because the numerator and the denominator describe different
populations, counted in different units, under different rules.

This notebook builds that ratio the way it is usually built, shows what it actually divides,
and then reports what the tables genuinely support instead.

## Learning Objectives

1. **Notice a unit mismatch that produces a plausible number.** Households over persons
   yields something between 0 and 1 that looks like a rate and is not one.
2. **Distinguish programme eligibility from a poverty threshold.** They are different rules,
   and the gap between them is measurable here.
3. **Choose between two tables that both claim to count SNAP** and differ by a definite
   amount.
4. **State a take-up rate's requirements** and conclude that this source does not meet them.

## Prerequisites

- D01 for the ACS poverty tables and the `E`/`M` convention.
- Handbook §3 (denominators, universes and partitions). This notebook is the sharpest case
  of it in the corpus.

## Data Sources

| table | concept | universe (Census wording) |
|---|---|---|
| `B22003` | Receipt of Food Stamps/SNAP by Poverty Status **for Households** | Households |
| `B19058` | Public Assistance Income **or** Food Stamps/SNAP **for Households** | Households |
| `B17001` | Poverty Status by Sex by Age | **Population for whom poverty status is determined** |

The universes above are quoted from the API's own group metadata, retrieved at build time.
Note the third: not "all people", but the subset for whom poverty status is even defined.

In [1]:
"""Configuration."""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "krl_tutorials_data").is_dir():
        sys.path.insert(0, str(_cand / "src"))
        break
else:
    raise SystemExit("Could not locate krl_tutorials_data from " + str(_here))

from krl_tutorials_data.sources import fetch_acs

STATE_FIPS = "51"
ACS_YEAR = 2023
#: The repository root, found by walking UP from wherever this runs. Provenance
#: used to be written to `Path.cwd() / "provenance"` in some notebooks and to a
#: repo-root-relative path in others, so the two conventions could not both be
#: satisfied by one working directory -- and a run from the wrong one silently
#: wrote sixteen records into a stray folder at the top of the repo.
REPO_ROOT = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (c / "src" / "krl_tutorials_data").is_dir())
OUT = REPO_ROOT / "projects" / "05-social-systems/D18_social_services_safety_net" / "provenance"
VARS = {
    "B22003_001E": "households", "B22003_001M": "households MOE",
    "B22003_002E": "received SNAP", "B22003_002M": "received SNAP MOE",
    "B22003_003E": "SNAP, income below poverty", "B22003_003M": "SNAP below poverty MOE",
    "B22003_004E": "SNAP, income at or above poverty", "B22003_004M": "SNAP above poverty MOE",
    "B19058_001E": "households (B19058 universe)",
    "B19058_002E": "public assistance income OR SNAP", "B19058_002M": "PA or SNAP MOE",
    "B17001_001E": "population poverty status determined",
    "B17001_002E": "persons below poverty", "B17001_002M": "persons below poverty MOE",
}
print(f"Virginia counties, ACS 5-year {ACS_YEAR - 4}-{ACS_YEAR}")

Virginia counties, ACS 5-year 2019-2023


In [2]:
"""Retrieve and pin provenance."""
sn, prov = fetch_acs(year=ACS_YEAR, variables=VARS,
                     geography="county:*", within=f"state:{STATE_FIPS}")
sn["fips"] = sn["state"] + sn["county"]
written = [prov.write(OUT / "d18_acs_safety_net.json")]
print(f"{len(sn)} counties and independent cities")
print()
print(prov.render()[:460])

133 counties and independent cities

Source      : U.S. Census Bureau - American Community Survey (acs/acs5)
Vintage     : 2023  (pinned)
Endpoint    : https://api.census.gov/data/2023/acs/acs5
Retrieved   : 2026-09-18T12:13:04+00:00
Shape       : 133 rows x 17 cols
Payload hash: 6e8ac1c5def45838...  (canonical-rows)
Data mode   : EMPIRICAL
Variables   :
              B22003_001E      households
              B22003_001M      households MOE
              B22003_002E      received SNAP
       


## 1. The ratio as it is usually built

Take households receiving SNAP, divide by people in poverty. Both come from the same release,
the same state, the same five-year window. The result is a number between 0 and 1 that reads
like a coverage rate.

In [3]:
"""Build it, then look at what it divided."""
snap_households = sn.B22003_002E.sum()
persons_in_poverty = sn.B17001_002E.sum()
ratio = snap_households / persons_in_poverty

print(f"households receiving SNAP     {snap_households:>10,.0f}   (B22003, unit: HOUSEHOLDS)")
print(f"persons below poverty         {persons_in_poverty:>10,.0f}   (B17001, unit: PERSONS)")
print(f"ratio                         {ratio:>10.3f}")
print()
print("Reported as a percentage this is "
      f"{100 * ratio:.1f}% -- a plausible-looking coverage rate.")
print()
print("It is not a rate. The numerator counts households and the denominator counts people,")
print("so the quantity has units of households per person. Virginia's average household")
print(f"contains about {sn.B17001_001E.sum() / sn.B22003_001E.sum():.2f} people for whom poverty status is")
print("determined, and that ratio varies across counties -- so the 'rate' moves with")
print("household size even where SNAP receipt and poverty are both unchanged.")

households receiving SNAP        293,101   (B22003, unit: HOUSEHOLDS)
persons below poverty            834,866   (B17001, unit: PERSONS)
ratio                              0.351

Reported as a percentage this is 35.1% -- a plausible-looking coverage rate.

It is not a rate. The numerator counts households and the denominator counts people,
so the quantity has units of households per person. Virginia's average household
contains about 2.53 people for whom poverty status is
determined, and that ratio varies across counties -- so the 'rate' moves with
household size even where SNAP receipt and poverty are both unchanged.


In [4]:
"""Does the unit mismatch actually distort the county ordering, or only the level?"""
from scipy import stats as st

bad_rate = sn.B22003_002E / sn.B17001_002E                 # households per person
household_size = sn.B17001_001E / sn.B22003_001E           # persons per household
rho = st.spearmanr(bad_rate, household_size).statistic

print(f"Spearman(the mismatched ratio, average household size) = {rho:+.3f}")
print(f"household size ranges {household_size.min():.2f} to {household_size.max():.2f} across counties")
print()
print("The association is real and in the direction the arithmetic predicts -- larger")
print("households mean fewer households per poor person -- but it is WEAK. The unit error")
print("does not visibly wreck the county ordering here.")
print()
print("That is the more important lesson, and it cuts against the instinct to check for")
print("damage before taking an error seriously. The quantity has the wrong units whether or")
print("not the ordering survives; a ratio of households to persons does not become a")
print("coverage rate because it happens to rank counties plausibly. An error that leaves")
print("the ranking intact is harder to catch, not less serious -- and had this correlation")
print("come out strong, it would have been caught long ago by someone puzzled at the map.")

Spearman(the mismatched ratio, average household size) = -0.200
household size ranges 1.94 to 3.27 across counties

The association is real and in the direction the arithmetic predicts -- larger
households mean fewer households per poor person -- but it is WEAK. The unit error
does not visibly wreck the county ordering here.

That is the more important lesson, and it cuts against the instinct to check for
damage before taking an error seriously. The quantity has the wrong units whether or
not the ordering survives; a ratio of households to persons does not become a
coverage rate because it happens to rank counties plausibly. An error that leaves
the ranking intact is harder to catch, not less serious -- and had this correlation
come out strong, it would have been caught long ago by someone puzzled at the map.


## 2. Eligibility is not the poverty line

The deeper problem is not units. `B22003` cross-tabulates SNAP receipt **with poverty status**,
so the overlap can be read directly rather than assumed — and the assumption most people carry
into this data does not hold.

In [5]:
"""Of households receiving SNAP, how many are actually below the poverty line?"""
below = sn.B22003_003E.sum()
above = sn.B22003_004E.sum()
total_snap = below + above

print(f"households receiving SNAP           {total_snap:>10,.0f}")
print(f"  income BELOW poverty              {below:>10,.0f}   {100 * below / total_snap:5.1f}%")
print(f"  income AT OR ABOVE poverty        {above:>10,.0f}   {100 * above / total_snap:5.1f}%")
print()
print("The majority of SNAP households in Virginia are not below the poverty line.")
print()
print("This is not an anomaly and not fraud. SNAP's gross income test runs to 130% of the")
print("federal poverty guidelines, its net test applies deductions the poverty measure does")
print("not, and it counts a different unit -- everyone who buys and prepares food together,")
print("which need not be a Census household. A programme built on one rule is being compared")
print("against a threshold built on another.")
print()
print("So 'SNAP recipients below poverty / all poor' understates coverage by discarding")
print(f"the {100 * above / total_snap:.0f}% above the line, while 'all SNAP / all poor' overstates the")
print("match between the two populations. Neither is a take-up rate.")

households receiving SNAP              293,101
  income BELOW poverty                 122,384    41.8%
  income AT OR ABOVE poverty           170,717    58.2%

The majority of SNAP households in Virginia are not below the poverty line.

This is not an anomaly and not fraud. SNAP's gross income test runs to 130% of the
federal poverty guidelines, its net test applies deductions the poverty measure does
not, and it counts a different unit -- everyone who buys and prepares food together,
which need not be a Census household. A programme built on one rule is being compared
against a threshold built on another.

So 'SNAP recipients below poverty / all poor' understates coverage by discarding
the 58% above the line, while 'all SNAP / all poor' overstates the
match between the two populations. Neither is a take-up rate.


## 3. Two tables that both count SNAP

`B19058` also reports SNAP for households — combined with cash public assistance. The two
are not interchangeable, and the difference is a definite quantity rather than noise.

In [6]:
"""How far apart are the two SNAP counts, and is the gap stable across counties?"""
b22 = sn.B22003_002E
b19 = sn.B19058_002E
gap = b19 - b22
pct = 100 * gap / b22.replace(0, np.nan)

print(f"B22003_002  SNAP only                    {b22.sum():>10,.0f}")
print(f"B19058_002  public assistance OR SNAP    {b19.sum():>10,.0f}")
print(f"difference                               {gap.sum():>+10,.0f}   ({100 * gap.sum() / b22.sum():+.1f}%)")
print()
print(f"Across counties the gap ranges from {pct.min():+.1f}% to {pct.max():+.1f}% "
      f"(median {pct.median():+.1f}%).")
print(f"Counties where B19058 is SMALLER than B22003: {(gap < 0).sum()} of {len(sn)}")
print()
print("B19058 is a superset by construction, so a negative gap would be a contradiction")
print("rather than a difference of definition. Checking for one is cheap and is the kind of")
print("test that catches a mis-specified variable before it reaches a chart.")
print()
print("The choice between them is substantive: B19058 answers 'how many households receive")
print("some means-tested cash or food assistance', B22003 answers 'how many receive SNAP'.")

B22003_002  SNAP only                       293,101
B19058_002  public assistance OR SNAP       320,496
difference                                  +27,395   (+9.3%)

Across counties the gap ranges from +0.0% to +64.3% (median +6.3%).
Counties where B19058 is SMALLER than B22003: 0 of 133

B19058 is a superset by construction, so a negative gap would be a contradiction
rather than a difference of definition. Checking for one is cheap and is the kind of
test that catches a mis-specified variable before it reaches a chart.

The choice between them is substantive: B19058 answers 'how many households receive
some means-tested cash or food assistance', B22003 answers 'how many receive SNAP'.


## 4. What a take-up rate would require

A take-up rate is *recipients ÷ eligible*. This source supplies neither term.

- The **numerator** would need recipients counted in the programme's own unit. `B22003`
  counts Census households, which SNAP's assistance unit does not map onto.
- The **denominator** would need the *eligible* population — a determination involving gross
  and net income tests, deductions, asset rules and categorical eligibility. Poverty status is
  not that, and §2 measured the gap.
- Both terms would need the same reference period. SNAP receipt here is "at any time in the
  past 12 months"; poverty status is an annual income measure. A household can appear in one
  and not the other without either being wrong.

What the data do support is stated below.

In [7]:
"""FINDINGS"""
print("FINDINGS")
print("=" * 78)
print(f"""
1. SNAP receipt reaches {100 * total_snap / sn.B22003_001E.sum():.2f}% of Virginia households
   ({total_snap:,.0f} of {sn.B22003_001E.sum():,.0f}). That is a rate: one unit, one universe,
   one table.

2. {100 * above / total_snap:.1f}% of SNAP households have income AT OR ABOVE the poverty line.
   Programme eligibility and the poverty threshold are different rules -- SNAP's
   gross test runs to 130% of the federal guidelines and its unit is people who
   buy and prepare food together, not a Census household.

3. Dividing SNAP households by persons in poverty produces {ratio:.3f}, which reads
   as {100 * ratio:.1f}% and is not a rate: its units are households per person. Its
   correlation with average household size is only rho = {rho:+.3f} -- weak, and in
   the direction the arithmetic predicts. The ordering mostly survives the
   error, which makes it harder to notice rather than less wrong.

4. B19058 and B22003 both count SNAP households and differ by {100 * gap.sum() / b22.sum():+.1f}%
   statewide, because B19058 also counts cash public assistance. B19058 is a
   superset by construction and is one in every county here, which is worth
   checking rather than assuming.

5. No take-up rate is computable from this source. It supplies neither recipients
   in the programme's unit nor the eligible population, and the two terms use
   different reference periods.
""")
print(f"Provenance written: {len(written)} record(s)")
for p in written:
    print(f"  {p.relative_to(REPO_ROOT)}")

FINDINGS

1. SNAP receipt reaches 8.81% of Virginia households
   (293,101 of 3,326,260). That is a rate: one unit, one universe,
   one table.

2. 58.2% of SNAP households have income AT OR ABOVE the poverty line.
   Programme eligibility and the poverty threshold are different rules -- SNAP's
   gross test runs to 130% of the federal guidelines and its unit is people who
   buy and prepare food together, not a Census household.

3. Dividing SNAP households by persons in poverty produces 0.351, which reads
   as 35.1% and is not a rate: its units are households per person. Its
   correlation with average household size is only rho = -0.200 -- weak, and in
   the direction the arithmetic predicts. The ordering mostly survives the
   error, which makes it harder to notice rather than less wrong.

4. B19058 and B22003 both count SNAP households and differ by +9.3%
   statewide, because B19058 also counts cash public assistance. B19058 is a
   superset by construction and is one in every 

## Key Takeaways

1. **Check units before arithmetic.** Households ÷ persons yields a number in the right range
   with the wrong meaning, and nothing raises an error.
2. **A programme's rule is not a statistical threshold.** Eligibility and poverty are separate
   constructions; here the majority of recipients sit above the line.
3. **When two tables count the same thing, find out how they differ** — and verify the
   direction the definition implies.
4. **Some questions are not answerable from a source.** Saying so is a finding, not a failure.

## Exercises

1. **Build the rate properly.** Recompute finding 1 per county and rank the top ten. Then
   apply handbook §7.1's overlap test using the published MOEs and report how many of those
   ten ranks are actually determined.
2. **Test the household-size artifact directly.** Regress the mismatched ratio on household
   size and on the correct SNAP rate. Which explains more of its variation?
3. **Find the counties where the two SNAP tables diverge most**, and check whether the gap
   tracks anything — cash assistance is much rarer than SNAP, so a large gap is informative.
4. **Bound the take-up rate anyway.** Using 130% of poverty as a crude eligibility proxy
   (`C17002` provides the ratio-to-poverty brackets), construct upper and lower bounds and
   state every assumption each bound requires.
5. **Change the state.** Repeat §2 elsewhere. Is the above-poverty share of SNAP households a
   Virginia fact or a national one? Predict first.

## References

- U.S. Census Bureau ACS group metadata for `B22003`, `B19058` and `B17001`, retrieved at
  build time — the universe wording in the Data Sources table is quoted from it.
- USDA Food and Nutrition Service, SNAP eligibility rules (gross and net income tests,
  household definition). *Not fetched in this build; the 130% gross-income figure is stated
  from standing knowledge and should be verified against the current rules before citation.*
- Handbook §3 (denominators, universes and partitions).

## Limitations

- **One state, one vintage.** The unit mismatch and the eligibility gap are general; the
  magnitudes are Virginia's 2019–2023.
- **Point estimates.** MOEs were retrieved and are pinned in provenance, but findings 1–4 use
  point estimates. Exercise 1 asks for the margin treatment on the ranking.
- **The 130% figure is unverified here.** It is used to explain the direction of the gap in
  §2, not to compute anything, and the measured quantity does not depend on it.
- **"Received SNAP in the past 12 months" is not "currently receives SNAP."** A household
  that received benefits for one month appears identically to one that received them for
  twelve, so this data cannot speak to benefit duration or spell length at all.
- **Not tested here:** whether the above-poverty share of SNAP households moved across
  vintages, which would separate a structural feature of the programme from a
  pandemic-period expansion.